In [1]:
# Core imports
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

# Display settings (optional)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Paths: notebooks/ is one level down from repo root
REPO_ROOT = Path("..").resolve()
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

# Create processed directory if it doesn't exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_RAW:", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)


REPO_ROOT: C:\Users\COMPUTER CARE\aeej1\JobPlatform
DATA_RAW: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\raw
DATA_PROCESSED: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed


In [2]:
JOBS_FILE = "jobs_dataset.csv"
RESUMES_FILE = "resumes_dataset.csv"

jobs_path = DATA_RAW / JOBS_FILE
resumes_path = DATA_RAW / RESUMES_FILE

# Safety checks
assert jobs_path.exists(), f"Jobs file not found: {jobs_path}"
assert resumes_path.exists(), f"Resumes file not found: {resumes_path}"

jobs = pd.read_csv(jobs_path)
resumes = pd.read_csv(resumes_path)

print("jobs shape:", jobs.shape)
print("resumes shape:", resumes.shape)


jobs shape: (1068, 7)
resumes shape: (1200, 14)


In [3]:
print("JOBS columns:")
print(list(jobs.columns))

print("\nRESUMES columns:")
print(list(resumes.columns))

display(jobs.head(3))
display(resumes.head(3))


JOBS columns:
['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords']

RESUMES columns:
['Name', 'Age', 'Gender', 'Education_Level', 'Field_of_Study', 'Degrees', 'Institute_Name', 'Graduation_Year', 'Experience_Years', 'Current_Job_Title', 'Previous_Job_Titles', 'Skills', 'Certifications', 'Target_Job_Description']


,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core fundamentals; ASP.NET; MVC; HTML; CSS; JavaScript basics; SQL Server; Entity Framework basics; LINQ; Visual Studio; Git; Unit Testing basics,Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; Support team in building ASP.NET MVC web applications; Write basic SQL queries and work with Enti...,.NET; C#; ASP.NET MVC; Entity Framework; SQL Server; LINQ; Visual Studio; Unit Testing
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTML; CSS; JavaScript basics; SQL Server; Entity Framework basics; NUnit basics,Write simple C# programs under guidance; Support development of ASP.NET MVC applications; Implement Razor views and front-end logic; Assist in database query writing; Participate in unit testing t...,.NET; C#; ASP.NET MVC; Entity Framework; SQL Server; Razor; Unit Testing
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTML; CSS; JavaScript basics; SQL Server; Git,Contribute to development of small modules; Assist in bug fixing and debugging; Learn and implement MVC patterns; Support database integration tasks; Understand version control basics; Work on min...,.NET; C#; ASP.NET MVC; SQL Server; Entity Framework; Git


,Name,Age,Gender,Education_Level,Field_of_Study,Degrees,Institute_Name,Graduation_Year,Experience_Years,Current_Job_Title,Previous_Job_Titles,Skills,Certifications,Target_Job_Description
0,Akash Pillai,30,Non-Binary,Master's,Cybersecurity,Master's in Cybersecurity,University of Pennsylvania,2025,0,NaN,NaN,"Node.js, JavaScript, Deep Learning, Statistics, SQL",Google Cloud Professional,Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.
1,Charlotte Taylor,27,Non-Binary,Bachelor's,Electronics Engineering,Bachelor's in Electronics Engineering,Pune University,2019,5,Cybersecurity Engineer,NaN,"Spark, Kubernetes, Terraform, Natural Language Processing",TensorFlow Developer Certificate,Targeting a Cybersecurity Engineer position to utilize my educational background and experience to drive results and achieve career objectives.
2,James Zhou,45,Male,Bachelor's,Computer Science,Bachelor's in Computer Science,Amity University,2023,2,Prompt Engineer,NaN,"Data Analysis, Node.js, Machine Learning, Linux, Jenkins, Network Security, REST APIs","Microsoft Azure Fundamentals, Cisco Certified Network Associate, Oracle Certified Professional",Targeting a Prompt Engineer position to utilize my educational background and experience to drive results and achieve career objectives.


In [4]:
def missing_report(df, name):
    print(f"\n{name} — missing values")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print("No missing values detected.")
    else:
        print(missing)

missing_report(jobs, "JOBS")
missing_report(resumes, "RESUMES")



JOBS — missing values
Title    1
dtype: int64

RESUMES — missing values
Previous_Job_Titles    914
Current_Job_Title      441
Certifications         319
dtype: int64


In [5]:
print("\nJOBS dtypes")
print(jobs.dtypes)

print("\nRESUMES dtypes")
print(resumes.dtypes)



JOBS dtypes
JobID                object
Title                object
ExperienceLevel      object
YearsOfExperience    object
Skills               object
Responsibilities     object
Keywords             object
dtype: object

RESUMES dtypes
Name                      object
Age                        int64
Gender                    object
Education_Level           object
Field_of_Study            object
Degrees                   object
Institute_Name            object
Graduation_Year            int64
Experience_Years           int64
Current_Job_Title         object
Previous_Job_Titles       object
Skills                    object
Certifications            object
Target_Job_Description    object
dtype: object


In [6]:
def inspect_skills(df, col, name):
    print(f"\n{name} — skills inspection ({col})")
    print("Type:", type(df[col].iloc[0]))
    print("Sample values:")
    for i in range(3):
        print("-", df[col].iloc[i])

inspect_skills(jobs, "Skills", "JOBS")
inspect_skills(resumes, "Skills", "RESUMES")



JOBS — skills inspection (Skills)
Type: <class 'str'>
Sample values:
- C#; VB.NET basics; .NET Framework; .NET Core fundamentals; ASP.NET; MVC; HTML; CSS; JavaScript basics; SQL Server; Entity Framework basics; LINQ; Visual Studio; Git; Unit Testing basics
- C#; .NET Framework basics; ASP.NET; Razor; HTML; CSS; JavaScript basics; SQL Server; Entity Framework basics; NUnit basics
- C#; VB.NET basics; .NET Core; ASP.NET MVC; HTML; CSS; JavaScript basics; SQL Server; Git

RESUMES — skills inspection (Skills)
Type: <class 'str'>
Sample values:
- Node.js, JavaScript, Deep Learning, Statistics, SQL
- Spark, Kubernetes, Terraform, Natural Language Processing
- Data Analysis, Node.js, Machine Learning, Linux, Jenkins, Network Security, REST APIs


In [7]:
print("Jobs count:", len(jobs))
print("Unique job titles:", jobs["Title"].nunique())
print("Experience levels:", jobs["ExperienceLevel"].value_counts())

print("\nResumes count:", len(resumes))
print("Freshers (0 years):", (resumes["Experience_Years"] == 0).sum())
print("Experienced (>0 years):", (resumes["Experience_Years"] > 0).sum())


Jobs count: 1068
Unique job titles: 218
Experience levels: ExperienceLevel
Experienced         476
Fresher             363
Entry-Level          66
Senior-Level         66
Mid-Level            60
Senior               15
Lead                  7
Junior                5
Mid-level             5
Mid-Senior Level      3
Mid-Senior            2
Name: count, dtype: int64

Resumes count: 1200
Freshers (0 years): 441
Experienced (>0 years): 759
